In [ ]:
!pip install torch

In [ ]:
!pip install transformers datasets accelerate peft evaluate scikit-learn sentencepiece

In [ ]:
import torch, transformers, datasets, peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)

# A Device-Agnostic Helper

In [ ]:
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")       # Windows/Linux with NVIDIA GPU
    if torch.backends.mps.is_available():
        return torch.device("mps")        # Apple Silicon Mac
    return torch.device("cpu")            # Fallback, works everywhere

device = get_device()
print(f"Using device: {device}")

# Loading and Inspecting Data

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Yelp/yelp_review_full")

print(dataset)
print(dataset["train"][0])

In [ ]:
# Using your own data
# from datasets import load_dataset

# dataset = load_dataset(
#     "csv",
#     data_files={"train": "train.csv", "test": "test.csv"}
# )

# Splitting Data Properly

In [ ]:
split = dataset["train"].train_test_split(test_size=0.2, seed=42)
train_val = split["train"].train_test_split(test_size=0.1, seed=42)

train_dataset = train_val["train"]
val_dataset = train_val["test"]
test_dataset = split["test"]

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# Tokenization

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"   # swap for any model on the Hub
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# Dynamic Padding for Efficiency

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Extracting Embeddings with a Sentence Transformer

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")   # small, fast, CPU-friendly

texts = train_dataset["text"]
embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)

print(embeddings.shape)   # (num_examples, embedding_dim)

# Extracting Embeddings from Any Transformer Backbone

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()   # freeze in inference mode — no gradients, no dropout

def extract_embeddings(texts, batch_size=32):
    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = tokenizer(batch, padding=True, truncation=True,
                                max_length=256, return_tensors="pt").to(device)
            outputs = model(**inputs)
            # Mean-pool the last hidden state across tokens for a single vector per example
            last_hidden = outputs.last_hidden_state
            mask = inputs["attention_mask"].unsqueeze(-1).expand(last_hidden.size()).float()
            summed = torch.sum(last_hidden * mask, dim=1)
            counts = torch.clamp(mask.sum(dim=1), min=1e-9)
            pooled = summed / counts
            all_embeddings.append(pooled.cpu().numpy())
    return np.concatenate(all_embeddings, axis=0)

train_embeddings = extract_embeddings(train_dataset["text"])
val_embeddings = extract_embeddings(val_dataset["text"])

# Training the Head

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

classifier = LogisticRegression(max_iter=1000)
classifier.fit(train_embeddings, train_dataset["label"])

val_predictions = classifier.predict(val_embeddings)
print(f"Validation accuracy: {accuracy_score(val_dataset['label'], val_predictions):.4f}")
print(classification_report(val_dataset["label"], val_predictions))

In [ ]:
new_texts = [
    "The battery life on this laptop is amazing",
    "Customer support never responded to my email",
]

new_embeddings = extract_embeddings(new_texts)

predictions = classifier.predict(new_embeddings)
probabilities = classifier.predict_proba(
    new_embeddings
)

for text, pred, prob in zip(
    new_texts, predictions, probabilities
):
    print(f"Text: {text}")
    print(f"Predicted label: {pred}")
    print(f"Confidence: {prob.max():.4f}\n")

In [ ]:
import torch.nn as nn

class EmbeddingClassifier(nn.Module):
    def __init__(self, embedding_dim, num_classes, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# Partial Fine-Tuning (Layer Freezing)

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
)

model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        "distilbert-base-uncased",
        num_labels=5,
    )
).to(device)

# First, freeze everything
for param in model.parameters():
    param.requires_grad = False

# Then, selectively unfreeze the last N
# transformer layers plus the classification head
num_layers_to_unfreeze = 2

# DistilBERT's transformer blocks live
# under model.distilbert.transformer.layer
total_layers = len(
    model.distilbert.transformer.layer
)
for layer in model.distilbert.transformer.layer[
    total_layers - num_layers_to_unfreeze:
]:
    for param in layer.parameters():
        param.requires_grad = True

# Always unfreeze the classification head —
# it's randomly initialized and must
# learn from scratch
for param in model.classifier.parameters():
    param.requires_grad = True

for param in model.pre_classifier.parameters():
    param.requires_grad = True

# Sanity check: how many parameters
# will actually be trained?
trainable = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)
total = sum(
    p.numel() for p in model.parameters()
)
print(
    f"Trainable: {trainable:,} / {total:,} "
    f"({100 * trainable / total:.2f}%)"
)

In [ ]:
print(model)

# Full Fine-Tuning

## Loading the model

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "distilbert-base-uncased"
num_labels = 5   # e.g., 1-5 star ratings

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels
)


## Defining Metrics

In [ ]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels,
    )
    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="weighted",
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"],
    }

## Configuring and Running Training

In [ ]:
from transformers import (
    TrainingArguments,
    Trainer,
)

training_args = TrainingArguments(
    output_dir="./full-finetune-output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
    # disable external experiment trackers
    # unless you use one
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## Saving and Reloading

In [ ]:
trainer.save_model("./my-fine-tuned-model")
tokenizer.save_pretrained(
    "./my-fine-tuned-model"
)

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

# confirms that the fine-tuned model 
# can be loaded
reloaded_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        "./my-fine-tuned-model"
    )
)
reloaded_tokenizer = (
    AutoTokenizer.from_pretrained(
        "./my-fine-tuned-model"
    )
)

## Testing the fine-tuned model

In [ ]:
import torch

# put the model in evaluation mode
reloaded_model.eval()

def predict(text):
    inputs = reloaded_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
    )
    with torch.no_grad():
        outputs = reloaded_model(**inputs)

    logits = outputs.logits
    predicted_class = torch.argmax(
        logits, dim=-1
    ).item()

    return predicted_class

# try it on a sample review
sample = (
    # "The food was cold and the "
    # "service was terribly slow.",
    "This is a fantastic restaurant!"    
)
print(predict(sample))